# AIPerf Use Case 5 — Time-Sliced Analysis: Performance Over Time

A single summary table averages over the whole run and hides *when* things were slow. `--slice-duration N` buckets results into N-second windows so you can see warm-up, degradation, and load-spike behavior — the gist's demo caught a cold start (slice 0 TTFT 545 ms vs ~344 ms once warm).

Based on the [AIPerf office-hours gist](https://gist.github.com/BenHamm/31c648f7d7331c94c1f3a45859db6677); notes in `docs/reference/aiperf-office-hours.md`.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — workload and slice duration](#3-input--workload-and-slice-duration)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)

## 1. Overview

With `--slice-duration`, AIPerf additionally writes `profile_export_aiperf_timeslices.csv` / `.json`: per-window request counts and metric statistics. What it surfaces:

- **Warm-up / cold start** — first slice slower despite fewer requests (model/kernel warm-up, cache fills).
- **Degradation over time** — creeping TTFT/ITL from memory leaks, KV-cache fragmentation, GC pauses.
- **Load-pattern response** — when replaying a day-long trace with spikes, how latency tracked each spike.

This is directly relevant to the repo's **Sustained/Soak scenario** (`docs/scenarios/sustained-soak-load.md`, `model-selection/sustained-soak-load/`): the soak script varies wall-clock duration to catch exactly these effects, and `--slice-duration` is the built-in way to see them *within* a run instead of only in the end-of-run aggregate.

## 2. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf`). The office-hours gist pinned `release/0.3.0`; pin whatever version you use (repo convention: record the AIPerf version per run).
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- Tip: AIPerf's live dashboard is designed for a terminal. If it renders poorly inside Jupyter, check `aiperf profile --help` for a plain/simple UI option in your version.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH — install it before running the Test run section.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())


## 3. Input — workload and slice duration

Any workload works. Choose `--slice-duration` relative to run length — enough slices to see a trend, enough requests per slice for stable statistics:

| Run length | Suggested slice |
|---|---|
| ~1 min demo (gist) | 10 s |
| 20–60 min soak | 1–5 min |
| multi-hour soak | 10–15 min |

The gist used the Mooncake 5-min/5× slice (Use Case 3 notebook); this notebook defaults to synthetic traffic so it runs standalone.

## 4. Test run

In [ ]:
def run_aiperf(cmd):
    """Print and run an aiperf command from the repo root, streaming output into the notebook."""
    print("$ " + " \\\n    ".join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
    print(f"\nexit code: {result.returncode}")
    return result

# ---- Required ----------------------------------------------------------
MODEL = ""
URL = ""

SLICE_DURATION = 10        # seconds per bucket
INPUT_FILE = ""            # optional mooncake-format JSONL (e.g. the UC3 slice); empty = synthetic
CONCURRENCY = 10
REQUEST_COUNT = 300        # enough requests to fill several slices
TOKENIZER = ""
# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc5-timeslices"

assert MODEL and URL, "Set MODEL and URL above."

cmd = [
    "aiperf", "profile",
    "--model", MODEL, "--url", URL,
    "--endpoint-type", "chat", "--streaming",
    "--tokenizer", TOKENIZER or MODEL,
    "--artifact-dir", OUTPUT_DIR,
    "--slice-duration", str(SLICE_DURATION),
]
if INPUT_FILE:
    cmd += ["--input-file", INPUT_FILE, "--custom-dataset-type", "mooncake_trace", "--fixed-schedule"]
else:
    cmd += ["--concurrency", str(CONCURRENCY), "--request-count", str(REQUEST_COUNT),
            # NOTE: --synthetic-input-tokens-mean / --output-tokens-mean are the canonical
            # long names for the gist's --isl / --osl shorthands. Used here because they are
            # guaranteed across AIPerf versions and match this repo's scenario scripts
            # (e.g. --output-tokens-mean in model-selection/scripts/run_content_generation.sh).
            "--synthetic-input-tokens-mean", "1000", "--output-tokens-mean", "500"]
run_aiperf(cmd)


## 5. Results analysis

Load the timeslice export and plot the trend. (Column names can vary across AIPerf versions — the first cell shows the raw table so you can adjust the plotting cell if needed.)

In [ ]:
import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
slices_csv = next(artifact_dir.rglob("*timeslices*.csv"), None)
assert slices_csv is not None, f"No timeslices CSV under {artifact_dir} — was --slice-duration set?"

slices = pd.read_csv(slices_csv)
print(f"{len(slices)} slices from {slices_csv.name}")
slices


In [ ]:
import matplotlib.pyplot as plt


def find_col(df, *needles):
    for c in df.columns:
        name = c.lower()
        if all(n in name for n in needles):
            return c
    return None


ttft_col = find_col(slices, "first token", "avg") or find_col(slices, "ttft", "avg")
count_col = find_col(slices, "request count") or find_col(slices, "count")
tput_col = find_col(slices, "throughput")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in [(axes[0], ttft_col, "TTFT avg per slice (ms)"),
                       (axes[1], tput_col, "Throughput per slice"),
                       (axes[2], count_col, "Requests per slice")]:
    if col is not None:
        axes_x = range(len(slices))
        ax.plot(axes_x, slices[col], marker="o")
        ax.set_title(title)
        ax.set_xlabel(f"slice # ({SLICE_DURATION}s each)")
    else:
        ax.set_title(f"{title}: column not found — see raw table above")
plt.tight_layout()


In [ ]:
# Simple trend checks: cold start (first slice vs. steady state) and drift (late vs. mid run).
if ttft_col is not None and len(slices) >= 3:
    first = slices[ttft_col].iloc[0]
    steady = slices[ttft_col].iloc[1:].median()
    print(f"Slice 0 TTFT: {first:.0f} ms vs. steady-state median {steady:.0f} ms "
          f"({100 * (first - steady) / steady:+.0f}%)  -> cold start if strongly positive")

    half = len(slices) // 2
    early, late = slices[ttft_col].iloc[1:half + 1].median(), slices[ttft_col].iloc[half:].median()
    print(f"Early-half TTFT median: {early:.0f} ms, late-half: {late:.0f} ms "
          f"({100 * (late - early) / early:+.0f}%)  -> degradation if strongly positive")


### Notes

- The gist's warm-up finding — slice 0 slower with *fewer* requests — is the signature of cold start rather than load. If you benchmark for comparisons (Model Selection / Sizing), that's why the repo's scripts send warm-up requests first (`--warmup-request-count`).
- For soak runs, sustained upward TTFT/ITL drift across slices (with constant load) is the leak/fragmentation signal the Sustained/Soak scenario exists to catch — consider adding `--slice-duration` there when it's next revised.
- The per-request JSONL (Use Case 2 notebook) can reproduce any custom bucketing if the fixed slices don't fit your analysis.